# Camada Silver — `ecommerce_clientes` 

Lê a Bronze física em `az://squad1/bronze/ecommerce_clientes`, aplica as 10 regras de qualidade, grava **somente linhas válidas** na Silver física em `az://squad1/silver/ecommerce_clientes` e registra as falhas em `az://squad1/dq_monitoring_logs`.

Este notebook possui modo de reprocessamento para quando os arquivos já foram lidos anteriormente.

In [0]:
%run ../utils/utils

##  Imports e parâmetros

In [0]:


#  Carrega as funções utilitárias (gravar_delta, ler_delta, etc)


import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timezone
from functools import reduce

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_clientes"
TABELA_DQ = "dq_monitoring_logs"
DATA_EXECUCAO = datetime.now(timezone.utc)

print(f"Iniciando processamento Silver - Clientes - Run ID: {RUN_ID}")

## Leitura do Micro-lote e Tabelas de Referência (Joins)

In [0]:
# 1. Carrega a tabela Bronze de Clientes
try:
    df_bronze_clientes = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada.")

# 2. Isola o Micro-lote (Considerando Silver E Quarentena)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)

    if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
        df_quarentena = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
        df_processados = df_silver_atual.select("id_cliente") \
            .union(df_quarentena.select("id_cliente"))
    else:
        df_processados = df_silver_atual.select("id_cliente")

    df_micro_lote = df_bronze_clientes.join(df_processados, "id_cliente", "left_anti")
else:
    df_micro_lote = df_bronze_clientes

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote para processar: {qtd_novos}")

# =================================================================================
# 3. Leitura das Tabelas de Referência para validação de Chaves (Regras 7 e 8)
# CORREÇÃO: ambas agora usam obter_referencia_ids_silver_ou_bronze (definida
# no utils.py), priorizando a Silver da tabela referenciada e caindo para a
# Bronze quando a Silver ainda não existir.
# =================================================================================

df_enderecos_ref = obter_referencia_ids_silver_ou_bronze("ecommerce_enderecos", "id_cliente") \
    .withColumnRenamed("id_cliente", "id_cliente_tem_endereco")

df_pedidos_ref = obter_referencia_ids_silver_ou_bronze("ecommerce_pedidos", "id_cliente") \
    .withColumnRenamed("id_cliente", "id_cliente_tem_pedido")

print("Tabelas de referência para validação cruzada carregadas.")

## Leitura da Bronze e Aplicação das 10 Regras de Data Quality



In [0]:
if qtd_novos > 0:
    # Expressões regulares e listas para validação
    regex_email = r"^.+@.+\..+$"
    regex_uuid = r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"
    provedores_validos = ["gmail", "yahoo", "hotmail", "outlook", "uol", "bol", "terra"]

    # CORREÇÃO: janelas agora têm orderBy explícito por bronze_ingested_at,
    # para que "a primeira ocorrência" seja determinística (a mesma linha
    # sempre "ganha" entre execuções, em vez de depender de ordem arbitrária
    # de leitura do Spark). row_number() == 1 marca o sobrevivente; qualquer
    # linha com row_number() > 1 é duplicata e é reprovada.
    w_id_cliente = Window.partitionBy("id_cliente").orderBy(F.col("bronze_ingested_at").asc())
    w_email = Window.partitionBy("email").orderBy(F.col("bronze_ingested_at").asc())
    w_uuid = Window.partitionBy("uuid_cliente").orderBy(F.col("bronze_ingested_at").asc())

    # Prepara o DataFrame base aplicando as transformações de tipo e os Joins cruzados
    df_base = (df_micro_lote
        .withColumn("dt_cadastro_ts", F.col("dt_cadastro").cast("timestamp"))
        .withColumn("dt_atualizacao_ts", F.col("dt_ultima_atualizacao").cast("timestamp"))
        # CORREÇÃO: row_number() dentro de cada grupo, em vez de count(*).
        # O primeiro (row_X_no_grupo == 1) é o candidato a sobreviver.
        .withColumn("row_id_cliente_no_grupo", F.row_number().over(w_id_cliente))
        .withColumn("row_email_no_grupo", F.row_number().over(w_email))
        .withColumn("row_uuid_no_grupo", F.row_number().over(w_uuid))
        .join(df_enderecos_ref, df_micro_lote.id_cliente == df_enderecos_ref.id_cliente_tem_endereco, "left_outer")
        .join(df_pedidos_ref, df_micro_lote.id_cliente == df_pedidos_ref.id_cliente_tem_pedido, "left_outer"))

    # Extração do provedor de e-mail
    df_base = df_base.withColumn("provedor", F.split(F.split(F.col("email"), "@")[1], r"\.")[0])

    # Aplicação massiva das 10 regras
    df_silver_clientes = (df_base
        .withColumn(
            "r1_id_cliente_falhou",
            # Nulo/vazio sempre reprova; duplicata só reprova a partir da
            # SEGUNDA ocorrência (row_id_cliente_no_grupo > 1). A primeira
            # ocorrência é validada.
            F.col("id_cliente").isNull() | (F.col("id_cliente").cast("string") == "") | (F.col("row_id_cliente_no_grupo") > 1)
        )
        .withColumn(
            "r2_email_falhou",
            F.col("email").isNull() | (~F.col("email").rlike(regex_email)) | (F.col("row_email_no_grupo") > 1)
        )
        .withColumn("r3_nome_sobrenome_falhou", F.col("nome").isNull() | (F.trim(F.col("nome")) == "") | F.col("sobrenome").isNull() | (F.trim(F.col("sobrenome")) == ""))
        .withColumn("r4_senha_hash_falhou", F.col("senha_hash").isNull() | (F.length(F.col("senha_hash")) != 64))
        .withColumn("r5_dt_cadastro_falhou", F.col("dt_cadastro_ts").isNull() | (F.col("dt_cadastro_ts") > F.current_timestamp()))
        .withColumn("r6_dt_atualizacao_falhou", F.col("dt_cadastro_ts").isNull() | F.col("dt_atualizacao_ts").isNull() | (F.col("dt_atualizacao_ts") < F.col("dt_cadastro_ts")))
        .withColumn("r7_endereco_associado_falhou", F.col("id_cliente_tem_endereco").isNull())
        .withColumn("r8_pedido_associado_falhou", F.col("id_cliente_tem_pedido").isNull())
        .withColumn("r9_provedor_reconhecido_falhou", F.col("email").isNotNull() & (~F.col("provedor").isin(provedores_validos)))
        .withColumn(
            "r10_uuid_cliente_falhou",
            F.col("uuid_cliente").isNull() | (~F.col("uuid_cliente").rlike(regex_uuid)) | (F.col("row_uuid_no_grupo") > 1)
        ))

    # --- MODIFICAÇÃO SOLICITADA: UNIFICAÇÃO DAS REGRAS ---
    # Todas as 10 regras agora jogam a linha para a quarentena e geram log

    all_regras = [
        "r1_id_cliente_falhou", "r2_email_falhou", "r3_nome_sobrenome_falhou",
        "r4_senha_hash_falhou", "r5_dt_cadastro_falhou", "r6_dt_atualizacao_falhou",
        "r7_endereco_associado_falhou", "r8_pedido_associado_falhou",
        "r9_provedor_reconhecido_falhou", "r10_uuid_cliente_falhou"
    ]

    # Criando a condição lógica unificada (basta falhar em uma para mandar para a quarentena)
    condicao_falha_unificada = reduce(lambda a, b: a | b, [F.col(c) for c in all_regras])

    # Aplicando as colunas finais de controle (removida a coluna silver_tem_aviso)
    df_silver_clientes = (df_silver_clientes
        .withColumn("silver_linha_valida", ~condicao_falha_unificada)
        .withColumn("silver_processed_at", F.current_timestamp())
        .withColumn("silver_run_id", F.lit(RUN_ID)))

    print("Muralha de qualidade unificada para clientes estruturada com sucesso. "
          "R1/R2/R10: primeira ocorrência de cada duplicata é validada, as seguintes são reprovadas.")
else:
    print("Nenhum cliente novo encontrado para processamento.")

## Geração do Log Unificado (DQ Monitoring)

In [0]:
if qtd_novos > 0:
    catalogo_regras = [
        {"coluna": "r1_id_cliente_falhou", "regra": "R1_ID_CLIENTE_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_email_falhou", "regra": "R2_EMAIL_INVALIDO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r3_nome_sobrenome_falhou", "regra": "R3_NOME_SOBRENOME_VAZIO", "severidade": "Critica"},
        {"coluna": "r4_senha_hash_falhou", "regra": "R4_SENHA_HASH_TAMANHO_ERRADO", "severidade": "Critica"},
        {"coluna": "r5_dt_cadastro_falhou", "regra": "R5_DATA_CADASTRO_NULA_FUTURA", "severidade": "Critica"},
        {"coluna": "r6_dt_atualizacao_falhou", "regra": "R6_DATA_ATUALIZACAO_ANTERIOR_CADASTRO", "severidade": "Critica"},
        {"coluna": "r7_endereco_associado_falhou", "regra": "R7_CLIENTE_SEM_ENDERECO", "severidade": "Critica"},
        {"coluna": "r8_pedido_associado_falhou", "regra": "R8_CLIENTE_INATIVO_SEM_PEDIDO", "severidade": "Critica"},
        {"coluna": "r9_provedor_reconhecido_falhou", "regra": "R9_PROVEDOR_EMAIL_NAO_RECONHECIDO", "severidade": "Critica"},
        {"coluna": "r10_uuid_cliente_falhou", "regra": "R10_UUID_INVALIDO_DUPLICADO", "severidade": "Critica"}
    ]

    total_registros = df_silver_clientes.count()
    logs_list = []
    
    # CORREÇÃO: O 'for' agora possui exatamente 4 espaços de recuo, ficando dentro do escopo do 'if'
    for r in catalogo_regras:
        qtd_falhas = df_silver_clientes.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID, 
                TABELA_ALVO, 
                r["regra"], 
                "FAIL", 
                r["severidade"],
                int(qtd_falhas), 
                int(total_registros), 
                datetime.now(timezone.utc), 
                f"Bronze Delta ({TABELA_ALVO})"
            ))
            
    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    print("Logs gerados prontos para gravação:", df_dq_monitoring_logs_novos.count())
    display(df_dq_monitoring_logs_novos)
else:
    df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())
    print("Sem novos logs: lote vazio.")

## Gravação Final (Silver Clientes e Logs)

In [0]:
if qtd_novos > 0:
    # CORREÇÃO: "silver_tem_aviso" removido do mapeamento para evitar Schema Mismatch
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id"]
    
    # ---------------- 1. GRAVAÇÃO DOS VÁLIDOS ---------------- #
    df_silver_validos = (df_silver_clientes
        .filter(F.col("silver_linha_valida") == True)
        .select(*colunas_finais))
        
    qtd_validos = df_silver_validos.count()
    print(f"Registros aprovados para a Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos, camada="silver", tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=True
        )
        if sucesso_silver:
            print(f"Tabela Silver {TABELA_ALVO} atualizada com sucesso!")

    # ---------------- 2. GRAVAÇÃO DA QUARENTENA (DEDUPLICADA) ---------------- #
    df_silver_invalidos = (df_silver_clientes
        .filter(F.col("silver_linha_valida") == False)
        .select(*colunas_finais))
        
    if df_silver_invalidos.count() > 0:
        if delta_existe("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS):
            df_quarentena_historico = ler_delta("silver/quarentena", TABELA_ALVO, STORAGE_OPTIONS)
            df_quarentena_para_gravar = df_silver_invalidos.join(
                df_quarentena_historico.select("id_cliente"), 
                on="id_cliente", 
                how="left_anti"
            )
        else:
            df_quarentena_para_gravar = df_silver_invalidos

        qtd_novos_rejeitados = df_quarentena_para_gravar.count()
        if qtd_novos_rejeitados > 0:
            sucesso_quarentena = gravar_delta(
                df=df_quarentena_para_gravar, camada="silver/quarentena", tabela=TABELA_ALVO,
                storage_opts=STORAGE_OPTIONS, mode="append", particionar=False 
            )
            if sucesso_quarentena:
                print(f"Enviados {qtd_novos_rejeitados} registros novos para a quarentena.")
        else:
            print("Todos os registros reprovados já existiam na quarentena histórica.")

    # ---------------- 3. GRAVAÇÃO DOS LOGS NA RAIZ ---------------- #
    if 'df_dq_monitoring_logs_novos' in locals() and df_dq_monitoring_logs_novos.count() > 0:
        sucesso_logs = gravar_delta(
            df=df_dq_monitoring_logs_novos, camada="", tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS, mode="append", particionar=False
        )
        if sucesso_logs:
            print("Logs de qualidade consolidados na raiz!")
else:
    print("Rotina finalizada sem alterações físicas.")

##  Validação Final

In [0]:
print("===== VALIDAÇÃO FINAL =====")

# 1. Validação da tabela Silver principal
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))
else:
    print(f"A tabela Silver {TABELA_ALVO} ainda não existe no Data Lake.")

# 2. Validação da tabela de Logs de Qualidade (na raiz do Data Lake)
if delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
    df_logs_validacao = ler_delta(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS)
    
    # Filtra para mostrar apenas os logs referentes à tabela de endereços
    df_logs_filtrados = df_logs_validacao.filter(F.col("tabela") == TABELA_ALVO)
    
    print(f"Logs na {TABELA_DQ} para {TABELA_ALVO}:", df_logs_filtrados.count())
    display(df_logs_filtrados.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print(f"Tabela {TABELA_DQ} ainda não existe no Data Lake.")